In [ ]:
import time
notebook_start = time.perf_counter()

import  os
import  json
import  pandas as pd
import  numpy as np
import  thermoift.PLOT_SETTINGS as ps
from    thermoift import MLPostprocessing

In [ ]:
# Parameters — overridden by papermill
PLOT_FOLDER = "SLURM_Pbubble"
SEED        = 454015

In [ ]:
# Load predictions saved by the GPU compute job
metrics_path  = os.path.join(PLOT_FOLDER, "TabPFN_P_bubble_metrics.json")
pred_path     = os.path.join(PLOT_FOLDER, "TabPFN_P_bubble_predictions.csv")

with open(metrics_path) as f:
    metrics = json.load(f)

results_df = pd.read_csv(pred_path)

target          = metrics["target"]
features        = metrics["features"]
cv_r2_scores    = np.array(metrics["cv_r2_scores"])
cv_rmse_scores  = np.array(metrics["cv_rmse_scores"])

print(f"Loaded {len(results_df)} predictions from {pred_path}")
print(f"Target: {target}")
print(f"Features ({len(features)}): {features}")
print(f"CV R²  mean: {cv_r2_scores.mean():.6f}")
print(f"CV RMSE mean: {cv_rmse_scores.mean():.6f}")

In [ ]:
# Reconstruct split arrays from results_df
train_df = results_df[results_df["split"] == "train"]
test_df  = results_df[results_df["split"] == "test"]
val_df   = results_df[results_df["split"] == "val"]

y_train      = pd.Series(train_df["actual"].values,    index=train_df["idx"].values)
y_train_pred = train_df["predicted"].values
y_test       = pd.Series(test_df["actual"].values,     index=test_df["idx"].values)
y_test_pred  = test_df["predicted"].values
y_val        = pd.Series(val_df["actual"].values,      index=val_df["idx"].values)
y_val_pred   = val_df["predicted"].values

print(f"Train: {len(y_train)}, Test: {len(y_test)}, Val: {len(y_val)}")

In [ ]:
# Reconstruct MLPostprocessing
post = MLPostprocessing(
    y_true=y_test,
    y_pred=y_test_pred,
    target=target,
    feature_names=features,
    datasets={
        "train": (y_train, y_train_pred),
        "test":  (y_test,  y_test_pred),
        "val":   (y_val,   y_val_pred),
    },
)

In [ ]:
# Parity plot
post.plot_parity(model_name="TabPFN", save_path="TabPFN_P_bubble_parity_plot", folder=PLOT_FOLDER)

In [ ]:
# Residual distribution
post.plot_residual_distribution(save_path="TabPFN_P_bubble_residual_distribution", folder=PLOT_FOLDER)

In [ ]:
# Residuals vs Predicted
post.plot_residual_vs_predicted(save_path="TabPFN_P_bubble_residual_vs_predicted", folder=PLOT_FOLDER)

In [ ]:
post.print_summary()

In [ ]:
notebook_end = time.perf_counter()
elapsed_minutes = (notebook_end - notebook_start) / 60
print(f"Total notebook runtime: {elapsed_minutes:.2f} minutes")